# Synthetic Logistic Regression Benchmarks

Boomerang vs Sticky Boomerang on controlled targets where we know the ground truth.

| Experiment | Target | What it tests |
|---|---|---|
| **A. Dense** | All 10 coefficients nonzero | Continuous posterior — sticky should recover the Boomerang |
| **B. Sparse** | Only 3 of 10 nonzero | True zeros — sticky should find them, Boomerang cannot |
| **C. Scaling** | Dense, varying n | ESS/grad and ESS/sec vs dataset size |

All experiments use Gaussian prior (scale=1.0), PLI thinning, and warmup preconditioning.

In [ ]:
import os, sys
os.chdir('../..')
import numpy as np
from benchmarks_august.targets.logreg import logreg_synthetic, logreg
from benchmarks_august.samplers import build_sampler, build_kappa
from benchmarks_august.samplers.warmstart import warmup_reference
from sazz.samplers.boomerang_sampler.utils import resample_pdmp_path, resample_sticky_pdmp_path
from benchmarks_august.analysis.metrics import (
    sample_quality, logreg_performance, sampler_efficiency,
    sticky_ess, _ess_batch_means
)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

In [ ]:
# Shared settings
N = 10000
n_resample = 50000
burnin = 0.2
refresh_rate = 1.0

---
## Experiment A — Dense target (p=10, all coefficients nonzero)

All coefficients are nonzero, so the sticky sampler gains nothing from sparsity.
It should recover similar predictive performance to the plain Boomerang.

In [ ]:
# Data
target_dense = logreg_synthetic(n=500, p=10, sparsity='dense', seed=42,
                                prior={'kind': 'gaussian', 'scale': 1.0})
X_d, y_d = target_dense.data['X'], target_dense.data['y']
X_tr_d, X_te_d, y_tr_d, y_te_d = train_test_split(X_d, y_d, test_size=0.2, random_state=0)
target_dense_train = logreg(X_tr_d, y_tr_d, prior={'kind': 'gaussian', 'scale': 1.0})

# sklearn MAP reference
lr_d = LogisticRegression(C=1.0, penalty='l2', fit_intercept=False, max_iter=1000).fit(X_tr_d, y_tr_d)
print('True beta:', target_dense.true_params)
print('MAP:      ', lr_d.coef_[0])

In [ ]:
# Boomerang PLI
boom_d = build_sampler('boomerang_pli', target_dense_train, N=N, refresh_rate=refresh_rate)
warmup_reference(boom_d, n_rounds=3, n_pilot=500, tune_refresh=True)
boom_d.reset(N=N)
boom_d.sample_auto(diagnostics=False)

In [ ]:
# Sticky Boomerang PLI
kappa_d = build_kappa({'kind': 'uniform', 'gamma_prior': 0.5}, target_dense_train)
sticky_d = build_sampler('sticky_boomerang_pli', target_dense_train, N=N,
                         kappa=kappa_d, refresh_rate=refresh_rate, cold_start_threshold=0.05)
warmup_reference(sticky_d, n_rounds=3, n_pilot=500, tune_refresh=True)
sticky_d.reset(N=N)
sticky_d.sample_auto(diagnostics=False)

In [ ]:
# NUTS via PyMC
import time
import pymc as pm

start = time.perf_counter()
with pm.Model() as model_d:
    beta = pm.Normal("beta", mu=0, sigma=1, shape=X_tr_d.shape[1])
    logits = pm.math.dot(X_tr_d, beta)
    y_obs = pm.Bernoulli("y", logit_p=logits, observed=y_tr_d)
    trace_d = pm.sample(5000, tune=2000, cores=1, random_seed=42)
nuts_wall_d = time.perf_counter() - start
nuts_samples_d = trace_d.posterior["beta"].values.reshape(-1, X_tr_d.shape[1])
print(f"NUTS wall time: {nuts_wall_d:.1f}s")

### A — Results

In [ ]:
# Resample
_, x_boom_d = resample_pdmp_path(boom_d, n_samples=n_resample, burnin_frac=burnin)
_, x_sticky_d = resample_sticky_pdmp_path(sticky_d, n_samples=n_resample, burnin_frac=burnin)

# Diagnostics plots
fig1 = sample_quality(x_boom_d, sklearn_coefs=lr_d.coef_[0], label='Boomerang PLI (dense)', burnin_frac=0)
fig2 = sample_quality(x_sticky_d, sklearn_coefs=lr_d.coef_[0], label='Sticky PLI (dense)', burnin_frac=0)
fig3 = sample_quality(nuts_samples_d, sklearn_coefs=lr_d.coef_[0], label='NUTS (dense)', burnin_frac=0)

In [ ]:
# Predictive performance
logreg_performance(X_tr_d, y_tr_d, X_te_d, y_te_d, {
    "NUTS": nuts_samples_d,
    'Boomerang PLI': x_boom_d,
    'Sticky PLI': x_sticky_d,
}, burnin_frac=0)

In [ ]:
# Sampler efficiency
sampler_efficiency({'Boomerang PLI': boom_d, 'Sticky PLI': sticky_d})

In [ ]:
# Sticky-aware ESS (standard ESS is misleading for sticky samples)
print("Sticky PLI — ESS diagnostics:")
s = sticky_ess(x_sticky_d, burnin_frac=0)
print(f"  Mean inclusion ESS:  {s['mean_inclusion_ess']:.0f}")
print(f"  Model size ESS:      {s['model_size_ess']:.0f}")
print(f"  Mean active ESS:     {s['mean_active_ess']:.0f}")

---
## Experiment B — Sparse target (p=10, only 3 nonzero)

7 of 10 true coefficients are exactly zero. The sticky sampler should discover this
sparsity structure and place mass at zero, while the Boomerang samples a continuous
posterior that never reaches exact zeros.

In [ ]:
# Data
target_sparse = logreg_synthetic(n=500, p=10, sparsity='sparse', seed=42,
                                 prior={'kind': 'gaussian', 'scale': 1.0})
X_s, y_s = target_sparse.data['X'], target_sparse.data['y']
X_tr_s, X_te_s, y_tr_s, y_te_s = train_test_split(X_s, y_s, test_size=0.2, random_state=0)
target_sparse_train = logreg(X_tr_s, y_tr_s, prior={'kind': 'gaussian', 'scale': 1.0})

lr_s = LogisticRegression(C=1.0, penalty='l2', fit_intercept=False, max_iter=1000).fit(X_tr_s, y_tr_s)
print('True beta:', target_sparse.true_params)
print('MAP:      ', lr_s.coef_[0])

In [ ]:
# Boomerang PLI
boom_s = build_sampler('boomerang_pli', target_sparse_train, N=N, refresh_rate=refresh_rate)
warmup_reference(boom_s, n_rounds=3, n_pilot=500, tune_refresh=True)
boom_s.reset(N=N)
boom_s.sample_auto(diagnostics=False)

In [ ]:
# Sticky Boomerang PLI
kappa_s = build_kappa({'kind': 'uniform', 'gamma_prior': 0.5}, target_sparse_train)
sticky_s = build_sampler('sticky_boomerang_pli', target_sparse_train, N=N,
                         kappa=kappa_s, refresh_rate=refresh_rate, cold_start_threshold=0.05)
warmup_reference(sticky_s, n_rounds=3, n_pilot=500, tune_refresh=True)
sticky_s.reset(N=N)
sticky_s.sample_auto(diagnostics=False)

In [ ]:
start = time.perf_counter()
with pm.Model() as model_s:
    beta = pm.Normal("beta", mu=0, sigma=1, shape=X_tr_s.shape[1])
    logits = pm.math.dot(X_tr_s, beta)
    y_obs = pm.Bernoulli("y", logit_p=logits, observed=y_tr_s)
    trace_s = pm.sample(5000, tune=2000, cores=1, random_seed=42)
nuts_wall_s = time.perf_counter() - start
nuts_samples_s = trace_s.posterior["beta"].values.reshape(-1, X_tr_s.shape[1])
print(f"NUTS wall time: {nuts_wall_s:.1f}s")

### B — Results

In [ ]:
# Resample
_, x_boom_s = resample_pdmp_path(boom_s, n_samples=n_resample, burnin_frac=burnin)
_, x_sticky_s = resample_sticky_pdmp_path(sticky_s, n_samples=n_resample, burnin_frac=burnin)

# Diagnostics plots
fig4 = sample_quality(x_boom_s, sklearn_coefs=lr_s.coef_[0], label='Boomerang PLI (sparse)', burnin_frac=0)
fig5 = sample_quality(x_sticky_s, sklearn_coefs=lr_s.coef_[0], label='Sticky PLI (sparse)', burnin_frac=0)
fig6 = sample_quality(nuts_samples_s, sklearn_coefs=lr_s.coef_[0], label='NUTS (sparse)', burnin_frac=0)

In [ ]:
# Predictive performance
logreg_performance(X_tr_s, y_tr_s, X_te_s, y_te_s, {
    "NUTS": nuts_samples_s,
    'Boomerang PLI': x_boom_s,
    'Sticky PLI': x_sticky_s,
}, burnin_frac=0)

In [ ]:
# Sampler efficiency
sampler_efficiency({'Boomerang PLI': boom_s, 'Sticky PLI': sticky_s})

In [ ]:
# Sticky-aware ESS
print("Sticky PLI (sparse) — ESS diagnostics:")
s = sticky_ess(x_sticky_s, burnin_frac=0)
print(f"  Mean inclusion ESS:  {s['mean_inclusion_ess']:.0f}")
print(f"  Model size ESS:      {s['model_size_ess']:.0f}")
print(f"  Mean active ESS:     {s['mean_active_ess']:.0f}")
print(f"  Per-coordinate inclusion ESS: {np.round(s['inclusion_ess'], 0)}")

---
## Experiment C — Scaling with dataset size

How does the Boomerang PLI scale as we increase the number of observations?
We measure ESS/gradient-eval and ESS/wall-second for a dense target with p=10.

In [ ]:
n_obs_list = [100, 200, 500, 1000, 2000, 5000, 10000]
results_scaling = []

for n_obs in n_obs_list:
    print(f'\nn_obs = {n_obs}')
    
    target = logreg_synthetic(n=n_obs, p=10, sparsity='dense', seed=42,
                              prior={'kind': 'gaussian', 'scale': 1.0})
    X_sc, y_sc = target.data['X'], target.data['y']
    
    # --- Boomerang PLI ---
    boom = build_sampler('boomerang_pli', target, N=N, refresh_rate=refresh_rate)
    warmup_reference(boom, n_rounds=3, n_pilot=500, tune_refresh=True)
    boom.reset(N=N)
    boom.sample_auto(diagnostics=False)
    
    _, x_boom = resample_pdmp_path(boom, n_samples=n_resample, burnin_frac=burnin)
    ess_boom = np.array([_ess_batch_means(x_boom[:, i]) for i in range(10)])
    df_boom = boom.diagnostics_df
    
    # --- Sticky Boomerang PLI ---
    kappa = build_kappa({'kind': 'uniform', 'gamma_prior': 0.5}, target)
    sticky = build_sampler('sticky_boomerang_pli', target, N=N,
                           kappa=kappa, refresh_rate=refresh_rate, cold_start_threshold=0.05)
    warmup_reference(sticky, n_rounds=3, n_pilot=500, tune_refresh=True)
    sticky.reset(N=N)
    sticky.sample_auto(diagnostics=False)
    
    _, x_sticky = resample_sticky_pdmp_path(sticky, n_samples=n_resample, burnin_frac=burnin)
    s_ess = sticky_ess(x_sticky, burnin_frac=0)
    df_sticky = sticky.diagnostics_df
    
    # --- NUTS ---
    t0 = time.perf_counter()
    with pm.Model():
        beta = pm.Normal("beta", mu=0, sigma=1, shape=10)
        logits = pm.math.dot(X_sc, beta)
        pm.Bernoulli("y", logit_p=logits, observed=y_sc)
        trace = pm.sample(5000, tune=2000, cores=1, random_seed=42)
    nuts_wall = time.perf_counter() - t0
    nuts_samples = trace.posterior["beta"].values.reshape(-1, 10)
    ess_nuts = np.array([_ess_batch_means(nuts_samples[:, i]) for i in range(10)])
    
    results_scaling.append({
        'n_obs': n_obs,
        # Boomerang
        'boom_min_ess': ess_boom.min(),
        'boom_grad_evals': int(df_boom['rate_evals'].sum()),
        'boom_wall_sec': df_boom['wall_seconds'].sum(),
        'boom_ess_per_grad': ess_boom.min() / int(df_boom['rate_evals'].sum()),
        'boom_ess_per_sec': ess_boom.min() / df_boom['wall_seconds'].sum(),
        # Sticky
        'sticky_inclusion_ess': s_ess['mean_inclusion_ess'],
        'sticky_model_size_ess': s_ess['model_size_ess'],
        'sticky_grad_evals': int(df_sticky['rate_evals'].sum()),
        'sticky_wall_sec': df_sticky['wall_seconds'].sum(),
        # NUTS
        'nuts_min_ess': ess_nuts.min(),
        'nuts_wall_sec': nuts_wall,
        'nuts_ess_per_sec': ess_nuts.min() / nuts_wall,
    })
    
    r = results_scaling[-1]
    print(f'  Boom:   min ESS={r["boom_min_ess"]:.0f}, ESS/grad={r["boom_ess_per_grad"]:.4f}, ESS/sec={r["boom_ess_per_sec"]:.0f}')
    print(f'  Sticky: incl ESS={r["sticky_inclusion_ess"]:.0f}, model ESS={r["sticky_model_size_ess"]:.0f}')
    print(f'  NUTS:   min ESS={r["nuts_min_ess"]:.0f}, ESS/sec={r["nuts_ess_per_sec"]:.0f}, wall={r["nuts_wall_sec"]:.1f}s')

### C — Scaling plots

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ns = [r['n_obs'] for r in results_scaling]

ax = axes[0]
ax.plot(ns, [r['boom_ess_per_grad'] for r in results_scaling], 'o-', color='steelblue', label='Boomerang')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('n (observations)'); ax.set_ylabel('min ESS / gradient eval')
ax.set_title('Computational efficiency')
ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(ns, [r['boom_ess_per_sec'] for r in results_scaling], 'o-', color='steelblue', label='Boomerang')
ax.plot(ns, [r['nuts_ess_per_sec'] for r in results_scaling], 's--', color='darkred', label='NUTS')
ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('n (observations)'); ax.set_ylabel('min ESS / second')
ax.set_title('Wall-clock efficiency')
ax.legend(); ax.grid(True, alpha=0.3)

fig.suptitle('Scaling with dataset size (p=10, dense)', fontsize=13)
plt.tight_layout()